# 02 因子分析
对 FU 主力合约计算全部因子，通过 IC/ICIR 检验有效性，分析因子相关性与分层收益。

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

from futuresquant.data.loader import FuturesDataLoader
from futuresquant.factors.engine import FactorEngine
from futuresquant.factors.technical import (
    ROC, MOM, RSI, BollingerBand, TSMomentum,
    MACross, MACD, ADX, PriceChannel,
    ATR, NormATR, HistoricalVolatility, VolatilityRatio,
    VolumeRatio, OBV, VWAP, OpenInterestChange,
)

DATA_DIR = r'I:\stock\FuturesQuant\raw_data\1min_FU'
loader = FuturesDataLoader(DATA_DIR)

# 使用 FU2210 作为研究标的
klines = loader.load('FU2210', start='2022-01-01', end='2022-10-31')
print(f'K线行数: {len(klines):,}')

## 1. 批量计算因子

In [ ]:
FACTORS = [
    ROC(20), MOM(20), RSI(14), BollingerBand(20), TSMomentum(5, 20),
    MACross(5, 20), MACD(12, 26, 9), ADX(14), PriceChannel(20),
    ATR(14), NormATR(14), HistoricalVolatility(240), VolatilityRatio(20, 120),
    VolumeRatio(20), OBV(20), VWAP(60, 14), OpenInterestChange(20),
]

engine = FactorEngine(FACTORS)
factor_df = engine.compute(klines)

print(f'因子矩阵: {factor_df.shape}')
factor_df.describe().round(4)

## 2. IC 分析（前视 N 根 bar 的收益相关性）

**Rank IC** = Spearman 相关系数(因子_t, 收益_{t → t+N})
- |IC| > 0.02 视为有效
- **ICIR** = IC均值 / IC标准差，衡量稳定性，>0.5 为佳

In [ ]:
def calc_ic(factor_df: pd.DataFrame, klines: pd.DataFrame,
            forward_bars: int = 60) -> pd.DataFrame:
    """计算每个因子的逐日 Rank IC，返回 (factor × date) DataFrame。"""
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)

    # 按交易日聚合：取每日最后一根 bar 的因子值和下一日收益
    daily_factor = factor_df.resample('1D').last()
    daily_ret = fwd_ret.resample('1D').last()

    ic_records = {}
    for col in daily_factor.columns:
        f = daily_factor[col]
        r = daily_ret
        aligned = pd.concat([f, r], axis=1).dropna()
        if len(aligned) < 5:
            continue
        # 滚动 20 天 Rank IC
        ic_series = aligned.apply(
            lambda row: stats.spearmanr(aligned[col], aligned['close']).correlation
            if len(aligned) > 1 else np.nan, axis=1
        )
        # 单值 IC（全样本 Spearman）
        rho, pval = stats.spearmanr(aligned[col], aligned['close'])
        ic_records[col] = {'IC': rho, 'p_value': pval}

    return pd.DataFrame(ic_records).T


def calc_rolling_ic(factor_series: pd.Series, klines: pd.DataFrame,
                    forward_bars: int = 60, window_days: int = 20) -> pd.Series:
    """计算单因子的滚动 Rank IC 序列（按日）。"""
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    daily_f = factor_series.resample('1D').last().dropna()
    daily_r = fwd_ret.resample('1D').last()
    combined = pd.concat([daily_f, daily_r], axis=1).dropna()
    combined.columns = ['factor', 'ret']

    rolling_ic = []
    for i in range(window_days, len(combined)):
        window = combined.iloc[i - window_days:i]
        rho, _ = stats.spearmanr(window['factor'], window['ret'])
        rolling_ic.append((combined.index[i], rho))

    if not rolling_ic:
        return pd.Series(dtype=float)
    times, vals = zip(*rolling_ic)
    return pd.Series(vals, index=pd.DatetimeIndex(times), name='rolling_IC')


ic_summary = calc_ic(factor_df, klines, forward_bars=60)
ic_summary['|IC|'] = ic_summary['IC'].abs()
ic_summary = ic_summary.sort_values('|IC|', ascending=False)
ic_summary.round(4)

In [ ]:
# IC 条形图
colors = ['#d62728' if x < 0 else '#1f77b4' for x in ic_summary['IC']]
fig = go.Figure(go.Bar(
    x=ic_summary.index, y=ic_summary['IC'],
    marker_color=colors,
    error_y=dict(type='data', array=[0]*len(ic_summary), visible=False),
))
fig.add_hline(y=0.02, line_dash='dash', line_color='green', annotation_text='IC=0.02')
fig.add_hline(y=-0.02, line_dash='dash', line_color='green')
fig.update_layout(title='各因子 Rank IC（forward 60bar）',
                   xaxis_title='因子', yaxis_title='IC', height=400)
fig.show()

## 3. 最佳因子的滚动 IC

In [ ]:
top_factor_name = ic_summary.index[0]
print(f'IC 最高因子: {top_factor_name}  IC={ic_summary.loc[top_factor_name, "IC"]:.4f}')

rolling_ic = calc_rolling_ic(factor_df[top_factor_name], klines, forward_bars=60)

fig = go.Figure()
fig.add_trace(go.Scatter(x=rolling_ic.index, y=rolling_ic.values,
                          mode='lines', name='20日滚动 IC',
                          line=dict(color='steelblue')))
fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.add_hline(y=rolling_ic.mean(), line_dash='dash', line_color='orange',
               annotation_text=f'均值={rolling_ic.mean():.3f}')
fig.update_layout(title=f'{top_factor_name} 滚动 Rank IC', height=350)
fig.show()

icir = rolling_ic.mean() / rolling_ic.std()
print(f'ICIR: {icir:.4f}   IC>0 比例: {(rolling_ic > 0).mean():.1%}')

## 4. 因子相关性热图

In [ ]:
corr = factor_df.dropna().corr(method='spearman').round(2)

fig = px.imshow(corr, text_auto=True, color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, aspect='auto',
                title='因子 Spearman 相关性矩阵')
fig.update_layout(height=600, width=700)
fig.show()

## 5. 因子分层回测
将每根 bar 的因子值按五分位分组，计算各组下 60 bar 的平均收益。
理想情况下，第 5 组 > 第 1 组（单调性）。

In [ ]:
def quantile_return(factor_series: pd.Series, klines: pd.DataFrame,
                    forward_bars: int = 60, n_quantiles: int = 5) -> pd.DataFrame:
    """因子分层：计算各分位组的平均前瞻收益。"""
    fwd_ret = klines['close'].pct_change(forward_bars).shift(-forward_bars)
    df = pd.concat([factor_series, fwd_ret], axis=1).dropna()
    df.columns = ['factor', 'fwd_ret']

    # 按日截面分位（避免时序污染）
    daily_last = df.resample('1D').last().dropna()
    daily_last['quantile'] = pd.qcut(daily_last['factor'], n_quantiles,
                                      labels=False, duplicates='drop') + 1
    return daily_last.groupby('quantile')['fwd_ret'].agg(['mean', 'std', 'count'])


# 对 IC 最高的几个因子做分层
top3 = ic_summary.index[:3].tolist()
fig = make_subplots(rows=1, cols=len(top3),
                    subplot_titles=[f'{f}\nIC={ic_summary.loc[f,"IC"]:.3f}' for f in top3])

for col_i, fname in enumerate(top3):
    qret = quantile_return(factor_df[fname], klines)
    fig.add_trace(
        go.Bar(x=[f'Q{int(q)}' for q in qret.index],
               y=qret['mean'] * 100,
               error_y=dict(type='data', array=qret['std'] * 100),
               name=fname, showlegend=False),
        row=1, col=col_i + 1
    )

fig.update_yaxes(title_text='平均收益 (%)', row=1, col=1)
fig.update_layout(title='因子分层平均收益（5分位，forward 60bar）', height=380)
fig.show()

## 6. 因子自相关性（衰减速度）

In [ ]:
lags = range(1, 61)  # 1 ~ 60 分钟 lag

fig = go.Figure()
for fname in top3:
    f = factor_df[fname].dropna()
    autocorr = [f.autocorr(lag) for lag in lags]
    fig.add_trace(go.Scatter(x=list(lags), y=autocorr, name=fname, mode='lines'))

fig.add_hline(y=0, line_color='black', line_width=0.8)
fig.update_layout(
    title='因子自相关性衰减（衰减越慢 → 因子持续时间越长）',
    xaxis_title='Lag (分钟)', yaxis_title='自相关系数', height=380
)
fig.show()